## ⚙️ Environment Setup (run once)

A maioria dos serviços AgentCore (Gateway, Memory, Runtime, Registry) requer
versões recentes de `boto3`. Esta cislula instala/atualiza tudo o que o
workshop precisa.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 03.1 — Cedar Language Basics

## Overview

[Cedar](https://www.cedarpolicy.com/) is a linguagem de policies que o
AgentCore Policy Engine usa. Sintaxe básica em uma linha:

```cedar
permit(principal, action, resource) when { conditions };
```

We will olhar as 9 policies que we will usar (P0-P8) e entender cada uma.

## Tutorial Details

| Information | Details |
|---|---|
| Tutorial type | Interactive (read-only) |
| AgentCore components | Policy |
| Complexity | Medium |
| SDK | — |
| Estimated time | 10 minutes |

## Prerequisites

Nenhum — este notebook is só leitura/teoria. Os nexts vão criar e testar.

## Estrutura de uma policy Cedar

```cedar
permit(                                  # ou forbid(
  principal is AgentCore::OAuthUser,     # quem (autenticado via JWT)
  action == AgentCore::Action::"...",    # o quê (a tool MCP sendo chamada)
  resource == AgentCore::Gateway::"...", # onde (o gateway)
)
when {                                    # condições adicionais (opcional)
  principal.getTag("cognito:groups") like "*operators*"
};
```

3 elementos obrigatórios: **principal**, **action**, **resource**.
Conditions in the `when` are optional and based on principal tags,
parâmetros da chamada (`context.input`), etc.

## As 9 policies do workshop

In [ ]:
import sys
sys.path.insert(0, "..")
from pathlib import Path

policies_dir = Path("../shared/policies/utility")
for cedar_file in sorted(policies_dir.glob("*.cedar")):
    print(f"\n{'='*60}")
    print(f"  {cedar_file.shas}")
    print('='*60)
    print(cedar_file.read_text())

## Cheat sheet das 9 policies

| Policy | Tipo | What it does |
|---|---|---|
| **P0** | permit | Tools without restriction (create_work_order, contracts, asset_history) — any authenticated user |
| **P1** | permit | Operadores podem ver grid e blackouts |
| **P2** | permit | Managers podem aprovar work orders |
| **P3** | forbid | Operadores **explicitamente** proibidos de aprovar (defense in depth) |
| **P4** | permit | Billing isolado — só governance/billing |
| **P5** | permit | Managers podem gerar relatórios regulatórios |
| **P6** | forbid | submit_to_regulator BLOQUEADO para todos (four-eyes) |
| **P7** | permit | Semantic search liberada |
| **P8** | forbid | create_work_order com priority=high requer manager (context-based!) |

## 🎓 What you learned

- Sintaxe Cedar (permit/forbid + 3 elementos + when)
- Como tags do principal (cognito:groups) viram filtros
- Diferença entre identity-based (P1-P5) e context-based (P8)

## Next

➡️ [03.2 — Attach Policies and Enforce](./02-attach-policies-and-enforce.ipynb)